# DeFi Yield Analysis Engine Demo

This notebook demonstrates the DeFi Yield Analysis Engine, which aggregates yield data from multiple Algorand DeFi protocols including Algofi, Folks Finance, and others.

## Features Demonstrated:
- Algofi lending/borrowing rates analysis
- Folks Finance pool yields with rewards
- Multi-protocol yield aggregation
- Asset-specific yield comparison
- Risk-adjusted yield calculations


In [ ]:
# Setup and imports
import sys
import asyncio
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime
import seaborn as sns

# Add the engine path
engine_path = Path.cwd().parent / "defi-yield-analysis"
sys.path.insert(0, str(engine_path))

from core.algofi_yields import AlgofiYieldsAnalyzer
from core.folks_finance_yields import FolksFinanceYieldsAnalyzer
from core.yield_aggregator import YieldAggregator

print("✅ Imports successful!")
print(f"Engine path: {engine_path}")

# Set style for plots
plt.style.use('default')
sns.set_palette("husl")

## 1. Initialize DeFi Protocol Analyzers

In [ ]:
# Initialize the analyzers
algofi_analyzer = AlgofiYieldsAnalyzer()
folks_analyzer = FolksFinanceYieldsAnalyzer()
yield_aggregator = YieldAggregator()

print("DeFi protocol analyzers initialized successfully!")

## 2. Algofi Protocol Analysis

In [ ]:
# Analyze Algofi yields
algofi_data = await algofi_analyzer.analyze_algofi_yields()

print("=== Algofi Protocol Analysis ===")
print(f"Weighted Supply APY: {algofi_data.weighted_average_supply_apy:.2%}")
print(f"Weighted Borrow APY: {algofi_data.weighted_average_borrow_apy:.2%}")
print(f"Total Value Locked: ${algofi_data.total_value_locked:,.0f}")
print(f"Protocol Utilization: {algofi_data.protocol_utilization:.2%}")
print(f"Number of Markets: {len(algofi_data.markets)}")
print(f"Confidence Score: {algofi_data.confidence_score:.2f}")

# Display individual markets
print(f"\n--- Individual Markets ---")
for market in algofi_data.markets[:5]:  # Show top 5 by supply
    spread = market.borrow_apy - market.supply_apy
    print(f"{market.asset_name:6s}: Supply {market.supply_apy:.2%}, Borrow {market.borrow_apy:.2%}, Spread {spread:.2%}, Util {market.utilization_rate:.1%}")

## 3. Folks Finance Protocol Analysis

In [ ]:
# Analyze Folks Finance yields
folks_data = await folks_analyzer.analyze_folks_yields()

print("=== Folks Finance Protocol Analysis ===")
print(f"Weighted Supply APY: {folks_data.weighted_supply_apy:.2%}")
print(f"Weighted Borrow APY: {folks_data.weighted_borrow_apy:.2%}")
print(f"Total Rewards APY: {folks_data.total_rewards_apy:.2%}")
print(f"Protocol TVL: ${folks_data.protocol_tvl:,.0f}")
print(f"Average Utilization: {folks_data.average_utilization:.2%}")
print(f"Number of Pools: {len(folks_data.pools)}")
print(f"Confidence Score: {folks_data.confidence_score:.2f}")

# Display individual pools
print(f"\n--- Individual Pools ---")
for pool in folks_data.pools[:5]:  # Show top 5 by TVL
    total_yield = pool.supply_apy + pool.pool_rewards_apy
    tvl_usd = pool.total_deposited * pool.oracle_price
    print(f"{pool.asset_name:6s}: Supply {pool.supply_apy:.2%}, Rewards {pool.pool_rewards_apy:.2%}, Total {total_yield:.2%}, TVL ${tvl_usd:,.0f}")

## 4. Multi-Protocol Yield Aggregation

In [ ]:
# Aggregate yields across all protocols
aggregated_yields = await yield_aggregator.aggregate_yields()

print("=== Multi-Protocol Yield Aggregation ===")
print(f"Weighted Supply APY: {aggregated_yields.weighted_supply_apy:.2%}")
print(f"Weighted Borrow APY: {aggregated_yields.weighted_borrow_apy:.2%}")
print(f"Total Rewards APY: {aggregated_yields.total_rewards_apy:.2%}")
print(f"Market Size Weighted APY: {aggregated_yields.market_size_weighted_apy:.2%}")
print(f"Protocol Count: {aggregated_yields.protocol_count}")
print(f"Total Market TVL: ${aggregated_yields.total_market_tvl:,.0f}")
print(f"Yield Spread: {aggregated_yields.yield_spread:.2%}")
print(f"Volatility Estimate: {aggregated_yields.volatility_estimate:.2%}")
print(f"Average Confidence: {aggregated_yields.average_confidence:.2f}")

# Protocol breakdown
print(f"\n--- Protocol Breakdown ---")
for protocol in aggregated_yields.protocol_breakdown:
    print(f"{protocol.protocol_name:15s}: Supply {protocol.supply_apy:.2%}, Rewards {protocol.additional_rewards:.2%}, TVL ${protocol.total_tvl:,.0f}, Weight {protocol.weight:.1%}")

## 5. Asset-Specific Yield Comparison

In [ ]:
# Compare yields for specific assets across protocols
assets_to_analyze = ['ALGO', 'USDC', 'USDT']
asset_yields = {}

print("=== Asset-Specific Yield Comparison ===")

for asset in assets_to_analyze:
    asset_yield = await yield_aggregator.get_asset_specific_yields(asset)
    asset_yields[asset] = asset_yield
    
    print(f"\n{asset} Analysis:")
    print(f"  Weighted Average Yield: {asset_yield.weighted_average_yield:.2%}")
    print(f"  Best Protocol: {asset_yield.best_yield_protocol}")
    print(f"  Worst Protocol: {asset_yield.worst_yield_protocol}")
    print(f"  Yield Range: {asset_yield.yield_range:.2%}")
    print(f"  Confidence Score: {asset_yield.confidence_score:.2f}")
    
    print(f"  Protocol Yields:")
    for protocol, yield_val in asset_yield.protocol_yields.items():
        print(f"    {protocol:15s}: {yield_val:.2%}")

## 6. Yield Trends Analysis

In [ ]:
# Analyze yield trends over time
trends = await yield_aggregator.get_yield_trends(7)

if "error" not in trends:
    print("=== 7-Day Yield Trends Analysis ===")
    
    supply_trend = trends['supply_apy_trend']
    borrow_trend = trends['borrow_apy_trend']
    market_metrics = trends['market_metrics']
    
    print(f"\nSupply APY Trend:")
    print(f"  Direction: {supply_trend['direction']}")
    print(f"  Start APY: {supply_trend['start_apy']:.2%}")
    print(f"  End APY: {supply_trend['end_apy']:.2%}")
    print(f"  Change: {supply_trend['change']:+.2%}")
    print(f"  Volatility: {supply_trend['volatility']:.2%}")
    print(f"  Average: {supply_trend['average']:.2%}")
    
    print(f"\nBorrow APY Trend:")
    print(f"  Direction: {borrow_trend['direction']}")
    print(f"  Start APY: {borrow_trend['start_apy']:.2%}")
    print(f"  End APY: {borrow_trend['end_apy']:.2%}")
    print(f"  Change: {borrow_trend['change']:+.2%}")
    print(f"  Volatility: {borrow_trend['volatility']:.2%}")
    print(f"  Average: {borrow_trend['average']:.2%}")
    
    print(f"\nMarket Metrics:")
    print(f"  Average TVL: ${market_metrics['average_tvl']:,.0f}")
    print(f"  TVL Trend: {market_metrics['tvl_trend']}")
else:
    print(f"Trends analysis error: {trends['error']}")

## 7. Visualization: Protocol Comparison

In [ ]:
# Create comprehensive protocol comparison visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. Supply APY Comparison
protocols = [p.protocol_name for p in aggregated_yields.protocol_breakdown]
supply_apys = [p.supply_apy * 100 for p in aggregated_yields.protocol_breakdown]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

bars1 = ax1.bar(protocols, supply_apys, color=colors[:len(protocols)], alpha=0.8)
ax1.set_ylabel('Supply APY (%)', fontweight='bold')
ax1.set_title('Supply APY by Protocol', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

for bar, value in zip(bars1, supply_apys):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.05,
             f'{value:.2f}%', ha='center', va='bottom', fontweight='bold')

# 2. TVL Comparison
tvls = [p.total_tvl / 1000000 for p in aggregated_yields.protocol_breakdown]  # Convert to millions
bars2 = ax2.bar(protocols, tvls, color=colors[:len(protocols)], alpha=0.8)
ax2.set_ylabel('TVL (Millions USD)', fontweight='bold')
ax2.set_title('Total Value Locked by Protocol', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for bar, value in zip(bars2, tvls):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'${value:.1f}M', ha='center', va='bottom', fontweight='bold')

# 3. Asset-specific yields comparison
asset_comparison_data = []
for asset, data in asset_yields.items():
    for protocol, yield_val in data.protocol_yields.items():
        asset_comparison_data.append({
            'Asset': asset,
            'Protocol': protocol,
            'Yield': yield_val * 100
        })

df_assets = pd.DataFrame(asset_comparison_data)
if not df_assets.empty:
    asset_pivot = df_assets.pivot(index='Asset', columns='Protocol', values='Yield')
    asset_pivot.plot(kind='bar', ax=ax3, color=colors[:len(asset_pivot.columns)])
    ax3.set_ylabel('Yield (%)', fontweight='bold')
    ax3.set_title('Asset Yields by Protocol', fontsize=14, fontweight='bold')
    ax3.legend(title='Protocol', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax3.grid(axis='y', alpha=0.3)
    ax3.tick_params(axis='x', rotation=0)

# 4. Risk vs Yield scatter plot
# Use confidence scores as inverse risk measure
risk_scores = [(1 - p.confidence_score) * 100 for p in aggregated_yields.protocol_breakdown]
yield_scores = [p.supply_apy * 100 for p in aggregated_yields.protocol_breakdown]

scatter = ax4.scatter(risk_scores, yield_scores, s=tvls, c=colors[:len(protocols)], alpha=0.7)
ax4.set_xlabel('Risk Score (100 - Confidence %)', fontweight='bold')
ax4.set_ylabel('Supply APY (%)', fontweight='bold')
ax4.set_title('Risk vs Yield (Bubble size = TVL)', fontsize=14, fontweight='bold')
ax4.grid(alpha=0.3)

# Add protocol labels
for i, protocol in enumerate(protocols):
    ax4.annotate(protocol, (risk_scores[i], yield_scores[i]), 
                xytext=(5, 5), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nAggregated Metrics Summary:")
print(f"Total Market TVL: ${aggregated_yields.total_market_tvl:,.0f}")
print(f"Weighted Supply APY: {aggregated_yields.weighted_supply_apy:.2%}")
print(f"Market Yield Spread: {aggregated_yields.yield_spread:.2%}")

## 8. Competitive Analysis

In [ ]:
# Perform competitive analysis
algofi_spreads = await algofi_analyzer.compare_supply_borrow_spreads()
folks_competitive = await folks_analyzer.compare_with_competitors()

print("=== Competitive Analysis ===")

# Algofi spreads analysis
if "error" not in algofi_spreads:
    print(f"\nAlgofi Spreads Analysis:")
    print(f"  Average Spread: {algofi_spreads['average_spread']:.2%}")
    print(f"  Median Spread: {algofi_spreads['median_spread']:.2%}")
    print(f"  Total Markets: {algofi_spreads['total_markets']}")
    
    print(f"  Top 3 Spreads:")
    for i, spread_data in enumerate(algofi_spreads['spreads'][:3], 1):
        print(f"    {i}. {spread_data['asset']}: {spread_data['spread']:.2%} (Util: {spread_data['utilization_rate']:.1%})")

# Folks competitive analysis
if "error" not in folks_competitive:
    print(f"\nFolks Finance Competitive Position:")
    folks_data_comp = folks_competitive['protocols']['folks_finance']
    print(f"  Supply APY: {folks_data_comp['weighted_supply_apy']:.2%}")
    print(f"  Total Rewards APY: {folks_data_comp['total_rewards_apy']:.2%}")
    print(f"  Protocol TVL: ${folks_data_comp['protocol_tvl']:,.0f}")
    
    print(f"  Competitive Advantages:")
    for advantage in folks_data_comp['competitive_advantage']:
        print(f"    • {advantage}")
    
    market_leader = folks_competitive['market_leader_supply']
    print(f"  Market Leader (Supply APY): {market_leader[0]} ({market_leader[1]:.2%})")
    print(f"  Total Market TVL: ${folks_competitive['total_market_tvl']:,.0f}")

## 9. Risk-Adjusted Yield Analysis

In [ ]:
# Calculate risk-adjusted yields
print("=== Risk-Adjusted Yield Analysis ===")

risk_adjusted_data = []

for protocol in aggregated_yields.protocol_breakdown:
    # Calculate risk-adjusted yield (simple model)
    risk_free_rate = 0.02  # 2% risk-free rate assumption
    risk_premium = protocol.supply_apy - risk_free_rate
    risk_factor = 1 - protocol.confidence_score  # Higher confidence = lower risk
    
    # Sharpe-like ratio
    risk_adjusted_yield = risk_premium / max(risk_factor, 0.01)  # Avoid division by zero
    
    risk_adjusted_data.append({
        'protocol': protocol.protocol_name,
        'raw_yield': protocol.supply_apy,
        'risk_premium': risk_premium,
        'risk_factor': risk_factor,
        'risk_adjusted_yield': risk_adjusted_yield,
        'confidence': protocol.confidence_score,
        'tvl': protocol.total_tvl
    })

# Display risk-adjusted analysis
for data in sorted(risk_adjusted_data, key=lambda x: x['risk_adjusted_yield'], reverse=True):
    print(f"\n{data['protocol']:15s}:")
    print(f"  Raw Yield: {data['raw_yield']:.2%}")
    print(f"  Risk Premium: {data['risk_premium']:.2%}")
    print(f"  Risk Factor: {data['risk_factor']:.2%}")
    print(f"  Risk-Adjusted Yield: {data['risk_adjusted_yield']:.2f}")
    print(f"  Confidence Score: {data['confidence']:.2%}")
    print(f"  TVL: ${data['tvl']:,.0f}")

# Create risk-adjusted visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Risk-adjusted yield comparison
protocols_ra = [d['protocol'] for d in risk_adjusted_data]
ra_yields = [d['risk_adjusted_yield'] for d in risk_adjusted_data]

bars = ax1.bar(protocols_ra, ra_yields, color=colors[:len(protocols_ra)], alpha=0.8)
ax1.set_ylabel('Risk-Adjusted Yield Score', fontweight='bold')
ax1.set_title('Risk-Adjusted Yield Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

for bar, value in zip(bars, ra_yields):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'{value:.1f}', ha='center', va='bottom', fontweight='bold')

# Confidence vs TVL scatter
confidences = [d['confidence'] * 100 for d in risk_adjusted_data]
tvls_millions = [d['tvl'] / 1000000 for d in risk_adjusted_data]

scatter = ax2.scatter(confidences, tvls_millions, s=150, c=colors[:len(risk_adjusted_data)], alpha=0.7)
ax2.set_xlabel('Confidence Score (%)', fontweight='bold')
ax2.set_ylabel('TVL (Millions USD)', fontweight='bold')
ax2.set_title('Confidence vs TVL', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

# Add protocol labels
for i, protocol in enumerate(protocols_ra):
    ax2.annotate(protocol, (confidences[i], tvls_millions[i]), 
                xytext=(5, 5), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Summary and Key Insights

In [ ]:
# Create comprehensive summary
print("=== DEFI YIELD ANALYSIS ENGINE SUMMARY ===")
print("="*55)

print(f"\n📊 AGGREGATED METRICS:")
print(f"   • Weighted Supply APY: {aggregated_yields.weighted_supply_apy:.2%}")
print(f"   • Weighted Borrow APY: {aggregated_yields.weighted_borrow_apy:.2%}")
print(f"   • Total Rewards APY: {aggregated_yields.total_rewards_apy:.2%}")
print(f"   • Total Market TVL: ${aggregated_yields.total_market_tvl:,.0f}")
print(f"   • Yield Spread: {aggregated_yields.yield_spread:.2%}")
print(f"   • Market Volatility: {aggregated_yields.volatility_estimate:.2%}")

print(f"\n🏦 PROTOCOL BREAKDOWN:")
for protocol in aggregated_yields.protocol_breakdown:
    print(f"   • {protocol.protocol_name:15s}: {protocol.supply_apy:.2%} APY, ${protocol.total_tvl:,.0f} TVL, {protocol.weight:.1%} weight")

print(f"\n🎯 TOP ASSET YIELDS:")
for asset, data in asset_yields.items():
    best_yield = max(data.protocol_yields.values()) if data.protocol_yields else 0
    print(f"   • {asset:6s}: {data.weighted_average_yield:.2%} avg, {best_yield:.2%} best ({data.best_yield_protocol})")

if "error" not in trends:
    print(f"\n📈 MARKET TRENDS (7 days):")
    print(f"   • Supply APY Trend: {trends['supply_apy_trend']['direction']} ({trends['supply_apy_trend']['change']:+.2%})")
    print(f"   • Borrow APY Trend: {trends['borrow_apy_trend']['direction']} ({trends['borrow_apy_trend']['change']:+.2%})")
    print(f"   • TVL Trend: {trends['market_metrics']['tvl_trend']}")
    print(f"   • Supply Volatility: {trends['supply_apy_trend']['volatility']:.2%}")

print(f"\n🏆 COMPETITIVE INSIGHTS:")
if "error" not in algofi_spreads:
    print(f"   • Algofi Average Spread: {algofi_spreads['average_spread']:.2%}")
if "error" not in folks_competitive:
    market_leader = folks_competitive['market_leader_supply']
    print(f"   • Market Leader: {market_leader[0]} ({market_leader[1]:.2%} APY)")
    print(f"   • Total Ecosystem TVL: ${folks_competitive['total_market_tvl']:,.0f}")

print(f"\n⚖️ RISK-ADJUSTED ANALYSIS:")
best_ra = max(risk_adjusted_data, key=lambda x: x['risk_adjusted_yield'])
print(f"   • Best Risk-Adjusted: {best_ra['protocol']} (Score: {best_ra['risk_adjusted_yield']:.1f})")
avg_confidence = sum(d['confidence'] for d in risk_adjusted_data) / len(risk_adjusted_data)
print(f"   • Average Confidence: {avg_confidence:.2%}")
total_tvl = sum(d['tvl'] for d in risk_adjusted_data)
print(f"   • Total Analyzed TVL: ${total_tvl:,.0f}")

print(f"\n✅ ENGINE STATUS:")
print(f"   • Protocols Analyzed: {aggregated_yields.protocol_count}")
print(f"   • Data Sources: {'Live APIs' if aggregated_yields.average_confidence > 0.8 else 'Simulated Data'}")
print(f"   • Aggregation Method: {aggregated_yields.metadata.get('calculation_method', 'Unknown')}")
print(f"   • Last Updated: {aggregated_yields.timestamp.strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*55)
print("🎉 DeFi Yield Analysis Engine Demo Complete!")